## テーブルの型を分析しやすいように変換

In [0]:
%run ../../config

### 型を変換した新テーブルを作成

In [0]:
# silverテーブルを作成
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {silver_audit_table_path}
    (
        `event_id` STRING,
        `event_time` TIMESTAMP,
        `action_name` STRING,
        `resource_name` STRING,
        `source_ip` STRING,
        `user` STRING,
        `user_email` STRING,
        `user_name` STRING,
        `request_params` STRING,
        _datasource STRING,
        _ingest_timestamp TIMESTAMP
    )
    """
)

# テーブル作り直し
spark.sql(
    f"""
    REPLACE TABLE {silver_audit_table_path} (
        `event_id` STRING,
        `event_time` TIMESTAMP,
        `action_name` STRING,
        `resource_name` STRING,
        `source_ip` STRING,
        `user` STRING,
        `user_email` STRING,
        `user_name` STRING,
        `request_params` STRING,
        _datasource STRING,
        _ingest_timestamp TIMESTAMP
    )
    """
)

## ブロンズテーブルを抽出し、型変換

In [0]:
df = spark.sql(
    f"""
    SELECT
        event_id,
        CAST(event_time AS TIMESTAMP) AS event_time,
        action_name,
        resource_name,
        source_ip,
        user,
        request_params,
        _datasource,
        _ingest_timestamp
    FROM {bronze_audit_table_path}
"""
)

In [0]:
# 処理後の結果を確認
df.display()

### クレンジング

- trim処理（値の空白除去）
- explodeでJSON展開
- null削除
- 重複削除

空白文字削除

**trim**  

> pyspark.sql.functions.trim(col: ColumnOrName)   
> 指定された文字列列の両端のスペースを切り取ります。  
> [Databricks｜pyspark.sql.functions.trim](https://api-docs.databricks.com/python/pyspark/latest/pyspark.sql/api/pyspark.sql.functions.trim.html)  

In [0]:

from pyspark.sql import functions as F

# trim処理
df = (
    df.withColumn("action_name", F.trim(F.col("action_name")))
    .withColumn("resource_name", F.trim(F.col("resource_name")))
    .withColumn("source_ip", F.trim(F.col("source_ip")))
    .withColumn("user", F.trim(F.col("user")))
    .withColumn("request_params", F.trim(F.col("request_params")))
)


**null削除**

- audit: `event_time`, `action_name`, `user_email` がnullの行を除外

In [0]:
df = df.filter(
    F.col("event_id").isNotNull()
    & F.col("event_time").isNotNull()
    & F.col("action_name").isNotNull()
    & F.col("user").isNotNull()
)


user JSON を展開（email/nameを列にする）

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

user_schema = StructType(
    [
        StructField("email", StringType(), True),
        StructField("name", StringType(), True),
    ]
)

df = (
    df.withColumn("user_struct", F.from_json(F.col("user"), user_schema))
    .withColumn("user_email", F.trim(F.col("user_struct.email")))
    .withColumn("user_name", F.trim(F.col("user_struct.name")))
    .drop("user_struct")
)

# user_email が必須のためNullは除外する（空白も除外）
df = df.filter(F.col("user_email").isNotNull() & (F.length(F.col("user_email")) > 0))

df.display()

重複削除

In [0]:
# 重複削除
# 指定キーの一意性を保証するが、同じevent_idが複数ある時にどれが残るかは分からない
df = df.dropDuplicates(["event_id"])

# 以下で、同じ event_id があったら 最新_ingestを残すこともできる
# from pyspark.sql import functions as F
# from pyspark.sql.window import Window

# w = Window.partitionBy("event_id").orderBy(F.col("_ingest_timestamp").desc())

# df = (
#   df.withColumn("_rn", F.row_number().over(w))
#     .filter(F.col("_rn") == 1)
#     .drop("_rn")
# )

In [0]:
# カラム順序を整列
df = df.select(
    "event_id",
    "event_time",
    "action_name",
    "resource_name",
    "source_ip",
    "user",
    "user_email",
    "user_name",
    "request_params",
    "_datasource",
    "_ingest_timestamp"
)


シルバーテーブルに書き込む

In [0]:
(
    df.write.format("delta")
    .mode("overwrite")
    .saveAsTable(silver_audit_table_path)
)

# spark.sql(f"optimize {silver_audit_table_path} zorder by (Timestamp)")
display(spark.sql(f"select * from {silver_audit_table_path}"))